In [10]:
import pandas as pd
from itertools import product

excel_file = "resultados.xlsx"

architectures = [
    [32], [64], [128], [264],
    [8, 4], [16, 8], [32, 16], [64, 32], [128, 64], [264, 128],
    [16, 8, 4], [32, 16, 8], [128, 64, 32], [264, 128, 64]
]

r_values = [0.01, 0.5, 0.9]

N_MODELS = 5  # número de seeds

df = pd.read_excel(excel_file)

comb = list(product(architectures, r_values))

for i, (arch, r) in enumerate(comb):
    
    start = i * N_MODELS
    end = start + N_MODELS
    
    df.loc[start:end-1, "Neurons"] = str(arch)
    df.loc[start:end-1, "r"] = r

df.to_excel("resultados.xlsx", index=False)

In [11]:
import numpy as np

TARGETS = ["Theta"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [12]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [13]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
display(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====


,Target,Dataset,Best_Model,Neurons,Best_R2
0,Theta,Train_1,model_128_Ld0.7_Lp0.3_r0.9_seed3,"[128, 64]",[0.921496055355566]
1,Theta,Train_2,model_264_Ld0.7_Lp0.3_r0.5_seed1,"[32, 16, 8]",[0.7334748408777436]
2,Theta,Val,model_32_16_8_Ld0.3_Lp0.7_r0.01_seed2,"[32, 16, 8]",[0.7890742897843109]
3,Theta,Test_1,model_128_64_Ld0.7_Lp0.3_r0.01_seed0,"[128, 64]",[0.8806270736209797]
4,Theta,Test_2,model_264_Ld0.7_Lp0.3_r0.5_seed1,"[32, 16, 8]",[0.7108046892620568]
5,Theta,Test_3,model_264_128_Ld0.7_Lp0.3_r0.5_seed2,"[264, 128]",[0.7833221017049696]


In [14]:
df_summary_top10["bestR2"] = best_only_df["Best_R2"].str.strip("[]").astype(float)
df_summary_top10["Neurons"] = best_only_df["Neurons"]
display(df_summary_top10)

,dataset,target,mean_r2,std_r2,mean_mse,std_mse,bestR2,Neurons
0,Train_1,Theta,0.916092,0.003138,0.019153,0.000716,0.921496,"[128, 64]"
1,Train_2,Theta,0.559959,0.061209,0.100463,0.013974,0.733475,"[32, 16, 8]"
2,Val,Theta,0.775802,0.006668,0.073375,0.002182,0.789074,"[32, 16, 8]"
3,Test_1,Theta,0.844516,0.026699,0.035718,0.006133,0.880627,"[128, 64]"
4,Test_2,Theta,0.584617,0.045547,0.098513,0.010802,0.710805,"[32, 16, 8]"
5,Test_3,Theta,0.737654,0.020996,0.086246,0.006903,0.783322,"[264, 128]"


- Entradas: SWdRef, SWeRef, Theta
- Saidas: X
- Trajetorias: ZZ + ZZRoted